# Verify both nozzle block encodings

All ten original code stages, the original 8 by 8 nozzle matrix with 21 entries, both Pauli and banded block encodings, fresh signed right-hand sides, and both unnormalized projections checked against the scaled matrix product.

The pinned CSR matrix is stored as plain numeric JSON. Pauli and diagonal decompositions are recomputed locally. Native synthesis prepares the data and selector amplitudes; the public Circuit composition applies Pauli SELECT or a bounded signed banded dilation, then unprepares the selector. These encodings have their own reported LCU scale and resources; no historical Gray-code gate count is claimed. Fixed local seeds make the generated right-hand sides reproducible. Absolute amplitudes are compared directly, with no sign correction or postselection renormalization.

## Verify an unnormalized block action, not only its state direction

The visible banded factory constructs explicit FlagQuantum Circuit kernels;
the projection wrapper chooses Pauli or banded encoding and executes it.
The host supplies the sparse matrix, band offsets and normalization. This
workflow does not yet ask native qfunc to infer an encoding from any matrix.
The direct Circuit code remains visible rather than being labeled automatic
high-level synthesis.

Try the two existing nonzero right-hand-side seeds and compare the full
selector-zero amplitudes with A @ rhs divided by scale. Do not normalize both
vectors first: that would hide a wrong block scale. Pauli and banded widths
and operation counts describe different encodings of the same declared action,
not proof of hardware speed or quantum advantage.


## Use the local FlagQuantum workflow

In [ ]:
import flagquantum as fq
import torch

from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.verify_block_encoding.overview.quantum import (
    CTYPE,
    bands,
    preparation,
)
from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.verify_block_encoding.overview.workflow import (
    CTYPE,
    Encoding,
    Projection,
    pauli_encoding,
)


def banded_encoding(
    matrix: torch.Tensor, rhs: torch.Tensor
) -> tuple[fq.Circuit, float]:
    offsets = bands(matrix)
    if offsets != (-1, 0, 1) or bool(matrix.imag.any()):
        raise ValueError(
            "this banded encoding requires the real source tridiagonal profile"
        )
    widths = [float(matrix.diagonal(offset).abs().max()) for offset in offsets]
    alpha = sum(widths)
    weights = preparation(
        torch.tensor([(w / alpha) ** 0.5 for w in widths] + [0], dtype=torch.float64)
    )
    matrix_parts = []
    for offset, scale in zip(offsets, widths, strict=True):
        diagonal = torch.zeros(8, dtype=torch.float64)
        shift = torch.zeros((8, 8), dtype=CTYPE)
        for col in range(8):
            row = col - offset
            if 0 <= row < 8:
                diagonal[col] = matrix[row, col].real / scale
            shift[row % 8, col] = 1
        d = torch.diag(diagonal).to(CTYPE)
        complement = torch.diag((1 - diagonal.square()).clamp_min(0).sqrt()).to(CTYPE)
        dilated = torch.cat(
            (torch.cat((d, complement), 1), torch.cat((complement, -d), 1)), 0
        )
        both = torch.zeros((16, 16), dtype=CTYPE)
        both[:8, :8] = both[8:, 8:] = shift
        matrix_parts.append(both @ dilated)
    matrix_parts.append(torch.eye(16, dtype=CTYPE))
    select = torch.zeros((64, 64), dtype=CTYPE)
    for index, part in enumerate(matrix_parts):
        group = slice(16 * index, 16 * (index + 1))
        select[group, group] = part
    torch.testing.assert_close(
        select.mH @ select, torch.eye(64, dtype=CTYPE), atol=1e-12, rtol=0
    )
    circuit = (
        fq.Circuit(6, dtype=CTYPE)
        .compose(weights, qubits=(0, 1))
        .compose(preparation(rhs), qubits=(3, 4, 5))
    )
    circuit.any(*range(6), matrix=select)
    circuit.compose(weights.adjoint(), qubits=(0, 1))
    return circuit, alpha


def project_preview(
    method: Encoding, matrix: torch.Tensor, vector: torch.Tensor
) -> Projection:
    if method not in ("pauli", "banded"):
        raise ValueError("choose pauli or banded")
    encoder = pauli_encoding if method == "pauli" else banded_encoding
    circuit, scale = encoder(matrix, vector)
    state = fq.run(circuit).statevector().flatten()
    values = state[:8]
    expected = matrix @ vector.to(CTYPE) / scale
    torch.testing.assert_close(values, expected, rtol=0, atol=1e-10)
    ir = circuit.to_ir()
    return Projection(
        method,
        ir.n_wires,
        len(ir.instructions),
        scale,
        values,
        expected,
        float((values - expected).abs().max()),
    )


from pathlib import Path

from IPython.display import Image, display


## Import preparation and verification tools

In [ ]:
from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.verify_block_encoding.overview.model import (
    bands,
    pauli_expansion,
    right_hand_side,
    source_matrix,
)
from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.verify_block_encoding.overview.plots import (
    draw,
)
from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.verify_block_encoding.overview.workflow import (
    project,
)


## Define the checked projection

In [ ]:
print("Apply the block encoding, then read only the selector-zero amplitudes.")


## Define the independent reference

In [ ]:
print("Compare the unnormalized projected amplitudes directly with A @ rhs / scale.")


## Choose the output directory

In [ ]:
folder = Path("artifacts/development/cfd-block-encoding")


## Load the original nozzle matrix

In [ ]:
matrix = source_matrix()
print(matrix.shape, matrix.count_nonzero().item())


## Regenerate and execute Pauli SELECT

In [ ]:
terms = pauli_expansion(matrix)
print("Pauli terms:", len(terms))
pauli_result = project_preview("pauli", matrix, right_hand_side(37))


## Compare the Pauli projection

In [ ]:
print("Pauli scale and error:", pauli_result.scale, pauli_result.max_error)
assert pauli_result.max_error < 1e-10
display(Image(filename=str(draw(pauli_result, folder / "pauli.png"))))


## Regenerate and execute the banded encoding

In [ ]:
print("Offsets:", bands(matrix))
banded_result = project_preview("banded", matrix, right_hand_side(91))


## Compare the banded projection

In [ ]:
print("Banded scale and error:", banded_result.scale, banded_result.max_error)
assert banded_result.max_error < 1e-10
display(Image(filename=str(draw(banded_result, folder / "banded.png"))))
